# Notebook 03 — Generating Synthetic Training Data with an LLM

Synthetic data generation lets you bootstrap a fine-tuning dataset when real labelled data is scarce.

In [ ]:
# !pip install openai tqdm pandas

## 1. Generate question-answer pairs with an LLM

In [ ]:
import openai
import json
import os
from tqdm import tqdm

client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "sk-placeholder"))

TOPICS = [
    "prompt injection attacks",
    "RAG pipeline architecture",
    "quantisation of neural networks",
    "reinforcement learning from human feedback",
]

def generate_qa(topic: str, n: int = 5) -> list[dict]:
    prompt = (
        f"Generate {n} question-answer pairs about: {topic}. "
        "Format each pair as JSON with keys 'question' and 'answer'. "
        "Output a JSON array."
    )
    # Stub for demo — replace with real API call
    stub = [{"question": f"What is {topic}?", "answer": f"A technique related to {topic}."} for _ in range(n)]
    return stub

all_pairs = []
for topic in tqdm(TOPICS):
    pairs = generate_qa(topic)
    for p in pairs:
        p["topic"] = topic
    all_pairs.extend(pairs)

print(f"Generated {len(all_pairs)} QA pairs")
print(all_pairs[0])

## 2. Deduplicate and quality-filter

In [ ]:
import pandas as pd

df = pd.DataFrame(all_pairs)
df = df.drop_duplicates(subset="question")
df = df[df["answer"].str.len() > 20]
print(f"After filtering: {len(df)} pairs")
df.head()

## 3. Save as JSONL

In [ ]:
import json

out_path = "../data/synthetic_qa.jsonl"
with open(out_path, "w") as f:
    for _, row in df.iterrows():
        f.write(json.dumps(row.to_dict()) + "\n")
print(f"Saved to {out_path}")

## 4. Self-consistency filtering

Generate each answer twice and keep only consistent ones.

In [ ]:
def answers_consistent(a: str, b: str, threshold: float = 0.8) -> bool:
    # Simple heuristic: character overlap > threshold
    set_a = set(a.lower().split())
    set_b = set(b.lower().split())
    if not set_a or not set_b:
        return False
    overlap = len(set_a & set_b) / max(len(set_a), len(set_b))
    return overlap >= threshold

# Demo
a = "Quantisation reduces model size by lowering the precision of weights."
b = "Quantization makes models smaller by using lower precision for weights."
print("Consistent:", answers_consistent(a, b))